
# Stage 4 — Improved Runnable BestModel Training Notebook (PCL Binary Classification)

This notebook is an improved, runnable version of a RoBERTa-based PCL classifier pipeline for **SemEval DPM Task 4 (binary PCL detection)**.

> **Important:** This notebook treats the 7-dim label vectors as *fine-grained PCL categories* and converts them to binary labels using:
> `label_bin = 1 if sum(category_vector) > 0 else 0`.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%cd /content/drive/MyDrive/COMP60035\ nlp/BestModel

/content/drive/MyDrive/COMP60035 nlp/BestModel


In [6]:
import os
import re
import ast
import json
import math
import random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import f1_score, precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

Configure hyperparameters, file paths, runtime options, and output folders for a reproducible run.

In [7]:
# Reproducibility
SEEDS = [42, 52, 62]
GLOBAL_SEED = 42

# Model & training
MODEL_NAME = "roberta-base"
MAX_LEN = 256
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1
LR = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1
DROPOUT = 0.1
MAX_GRAD_NORM = 1.0

# Imbalance handling
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.75
USE_WEIGHTED_SAMPLER = False

# Threshold optimisation
THRESHOLD_GRID = np.linspace(0.05, 0.95, 181)

# Runtime / output
RUN_NAME = "roberta_focal_threshold_seedensemble"
SAVE_DIRNAME = "output"

# Mixed precision (CUDA only)
USE_AMP = torch.cuda.is_available()

# Paths
BASE_DIR = Path(".").resolve()
PROJECT_DIR = BASE_DIR.parent
TRAIN_DIR = PROJECT_DIR / "train"
TEST_DIR = PROJECT_DIR / "test"
TSV_PATH = PROJECT_DIR / "dontpatronizeme_pcl.tsv"

RUN_ROOT = BASE_DIR / SAVE_DIRNAME
RUN_DIR = RUN_ROOT / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"

for p in [RUN_ROOT, RUN_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("BASE_DIR    :", BASE_DIR)
print("PROJECT_DIR :", PROJECT_DIR)
print("TRAIN_DIR   :", TRAIN_DIR)
print("TEST_DIR    :", TEST_DIR)
print("TSV_PATH    :", TSV_PATH)
print("device      :", device)

assert TSV_PATH.exists(), f"Missing TSV file: {TSV_PATH}"
assert (TRAIN_DIR / "train_semeval_parids-labels.csv").exists(), "Missing train split labels CSV"
assert (TRAIN_DIR / "dev_semeval_parids-labels.csv").exists(), "Missing dev split labels CSV"


BASE_DIR    : /content/drive/MyDrive/COMP60035 nlp/BestModel
PROJECT_DIR : /content/drive/MyDrive/COMP60035 nlp
TRAIN_DIR   : /content/drive/MyDrive/COMP60035 nlp/train
TEST_DIR    : /content/drive/MyDrive/COMP60035 nlp/test
TSV_PATH    : /content/drive/MyDrive/COMP60035 nlp/dontpatronizeme_pcl.tsv
device      : cuda


Define helper functions (seeding, label parsing, metrics utilities) used across the notebook.

In [8]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(GLOBAL_SEED)

def parse_label_vector(x) -> List[int]:
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return [0]*7
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            if isinstance(v, list):
                return [int(z) for z in v]
        except Exception:
            pass
    raise ValueError(f"Could not parse label vector: {x}")

def vec_to_binary(label_vec: List[int]) -> int:
    return int(sum(label_vec) > 0)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax_np(logits):
    logits = np.asarray(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)

def optimise_threshold(y_true: np.ndarray, p_pos: np.ndarray, grid=THRESHOLD_GRID):
    best = {"threshold": 0.5, "f1_pcl": -1.0, "precision": 0.0, "recall": 0.0}
    y_true = np.asarray(y_true).astype(int)
    p_pos = np.asarray(p_pos).astype(float)
    for t in grid:
        y_pred = (p_pos >= t).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        if f1 > best["f1_pcl"]:
            best = {"threshold": float(t), "f1_pcl": float(f1), "precision": float(p), "recall": float(r)}
    return best

def metrics_at_threshold(y_true: np.ndarray, p_pos: np.ndarray, threshold: float):
    y_pred = (np.asarray(p_pos) >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return {
        "threshold": float(threshold),
        "precision_pcl": float(p),
        "recall_pcl": float(r),
        "f1_pcl": float(f1),
        "macro_f1": float(macro_f1),
    }, y_pred

def save_predictions_txt(par_ids: List[int], preds: List[int], out_path: Path):
    # Common semeval format: one prediction per line in split order (not par_id)
    # We also save a CSV with par_id for traceability.
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for y in preds:
            f.write(f"{int(y)}\n")
    trace_path = out_path.with_suffix(".csv")
    pd.DataFrame({"par_id": par_ids, "pred": preds}).to_csv(trace_path, index=False)
    print(f"Saved: {out_path}")
    print(f"Saved trace CSV: {trace_path}")


## 3) Load DPM TSV and Official Train/Dev Split Labels

In [9]:
def load_dpm_tsv(path: Path) -> pd.DataFrame:
    cols = ["par_id", "art_id", "keyword", "country_code", "text", "label_ordinal"]
    # Uploaded TSV includes a 4-line disclaimer header
    df = pd.read_csv(path, sep="\t", header=None, skiprows=4, names=cols, engine="python")
    df["par_id"] = pd.to_numeric(df["par_id"], errors="coerce")
    df["label_ordinal"] = pd.to_numeric(df["label_ordinal"], errors="coerce")
    df = df.dropna(subset=["par_id", "text"]).copy()
    df["par_id"] = df["par_id"].astype(int)
    df["text"] = df["text"].astype(str)
    return df

dpm_df = load_dpm_tsv(TSV_PATH)
print("DPM TSV shape:", dpm_df.shape)
display(dpm_df.head(3))
print("Ordinal label distribution (from TSV):")
display(dpm_df["label_ordinal"].value_counts(dropna=False).sort_index())


DPM TSV shape: (10468, 6)


,par_id,art_id,keyword,country_code,text,label_ordinal
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0
2,3,@@16584954,immigrant,ie,White House press secretary Sean Spicer said t...,0


Ordinal label distribution (from TSV):


,count
label_ordinal,
0,8528
1,947
2,144
3,458
4,391


In [10]:
train_csv = TRAIN_DIR / "train_semeval_parids-labels.csv"
dev_csv = TRAIN_DIR / "dev_semeval_parids-labels.csv"

train_split = pd.read_csv(train_csv)
dev_split = pd.read_csv(dev_csv)

for split_name, split_df in [("train", train_split), ("dev", dev_split)]:
    assert {"par_id", "label"}.issubset(set(split_df.columns)), f"{split_name} CSV must contain par_id,label"
    split_df["par_id"] = pd.to_numeric(split_df["par_id"], errors="coerce").astype(int)
    split_df["label_vec"] = split_df["label"].apply(parse_label_vector)
    split_df["label_bin"] = split_df["label_vec"].apply(vec_to_binary)

print("train split shape:", train_split.shape)
print("dev split shape  :", dev_split.shape)

print("\nBinary label distribution from split CSVs:")
print("Train:")
display(train_split["label_bin"].value_counts().sort_index())
print("Dev:")
display(dev_split["label_bin"].value_counts().sort_index())


train split shape: (8375, 4)
dev split shape  : (2094, 4)

Binary label distribution from split CSVs:
Train:


,count
label_bin,
0,7581
1,794


Dev:


,count
label_bin,
0,1895
1,199


Merge the split labels with paragraph text from the main TSV and check for missing rows after merging. Note there's one row without text, we replace with "" to include it

In [11]:
text_cols = ["par_id", "art_id", "keyword", "country_code", "text"]

train_df = train_split.merge(dpm_df[text_cols], on="par_id", how="left", validate="one_to_one")
dev_df = dev_split.merge(dpm_df[text_cols], on="par_id", how="left", validate="one_to_one")

# Diagnose missing rows
missing_train_rows = train_df[train_df["text"].isna()].copy()
missing_dev_rows = dev_df[dev_df["text"].isna()].copy()

print("Missing train rows after merge:", len(missing_train_rows))
print("Missing dev rows after merge  :", len(missing_dev_rows))

if len(missing_train_rows) > 0:
    print("Missing train par_ids:", missing_train_rows["par_id"].tolist())

if len(missing_dev_rows) > 0:
    print("Missing dev par_ids:", missing_dev_rows["par_id"].tolist())

train_df["text"] = train_df["text"].fillna("")
dev_df["text"] = dev_df["text"].fillna("")

train_df = train_df[["par_id", "text", "label_bin", "label_vec", "keyword", "country_code"]].copy()
dev_df = dev_df[["par_id", "text", "label_bin", "label_vec", "keyword", "country_code"]].copy()

print("Final train_df:", train_df.shape)
print("Final dev_df  :", dev_df.shape)

print("\nTrain PCL rate:", train_df["label_bin"].mean())
print("Dev PCL rate  :", dev_df["label_bin"].mean())

display(train_df.head(2))
display(dev_df.head(2))

Missing train rows after merge: 0
Missing dev rows after merge  : 1
Missing dev par_ids: [8640]
Final train_df: (8375, 6)
Final dev_df  : (2094, 6)

Train PCL rate: 0.09480597014925374
Dev PCL rate  : 0.09503342884431709


,par_id,text,label_bin,label_vec,keyword,country_code
0,4341,"The scheme saw an estimated 150,000 children f...",1,"[1, 0, 0, 1, 0, 0, 0]",poor-families,gb
1,4136,Durban 's homeless communities reconciliation ...,1,"[0, 1, 0, 0, 0, 0, 0]",homeless,za


,par_id,text,label_bin,label_vec,keyword,country_code
0,4046,We also know that they can benefit by receivin...,1,"[1, 0, 0, 1, 0, 0, 0]",hopeless,us
1,1279,Pope Francis washed and kissed the feet of Mus...,1,"[0, 1, 0, 0, 0, 0, 0]",refugee,ng


Load and parse the official test file robustly (supports both full-row and ID-only formats), then prepare `test_df`.

In [12]:
TEST_SPLIT_PATH = PROJECT_DIR / "test" / "task4_test.tsv"
assert TEST_SPLIT_PATH.exists(), f"Missing: {TEST_SPLIT_PATH}"

# Try reading with header first (most common)
test_df = pd.read_csv(TEST_SPLIT_PATH, sep="\t", engine="python")

# If header isn't present / columns aren't as expected, fall back to no-header full-row format
if ("par_id" not in test_df.columns) or ("text" not in test_df.columns):
    raw = pd.read_csv(TEST_SPLIT_PATH, sep="\t", header=None, engine="python")

    # If the first row looks like a header row, drop it
    header_tokens = {"par_id", "paragraph_id", "art_id", "keyword", "country_code", "text"}
    first_row = [str(v).strip().lower() for v in raw.iloc[0].tolist()] if len(raw) > 0 else []
    if any(v in header_tokens for v in first_row):
        raw = raw.iloc[1:].reset_index(drop=True)

    # Expect full rows: par_id, art_id, keyword, country_code, text
    assert raw.shape[1] >= 5, f"Unexpected test.tsv shape: {raw.shape} (expected >=5 columns)"
    raw = raw.iloc[:, :5].copy()
    raw.columns = ["par_id", "art_id", "keyword", "country_code", "text"]
    test_df = raw

# Preserve exact IDs for submission
test_df["par_id"] = test_df["par_id"].astype(str)

# Ensure text exists (do NOT drop rows)
test_df["text"] = test_df["text"].fillna("")

# Keep only needed columns for inference; keep others if you want
test_df = test_df[["par_id", "text"]].copy()

print("Using test split file:", TEST_SPLIT_PATH)
print("Final test_df shape:", test_df.shape)
display(test_df.head(5))

Using test split file: /content/drive/MyDrive/COMP60035 nlp/test/task4_test.tsv
Final test_df shape: (3832, 2)


,par_id,text
0,t_0,"In the meantime , conservatives are working to..."
1,t_1,In most poor households with no education chil...
2,t_2,The real question is not whether immigration i...
3,t_3,"In total , the country 's immigrant population..."
4,t_4,"Members of the church , which is part of Ken C..."


## 5) Tokenizer + Dataset + Dataloaders

Initialize tokenizer, dataset class, and dataloaders for train/dev/test batching.

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class PCLDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = 128, has_labels: bool = True):
        self.df = df.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row["text"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["par_id"] = str(row["par_id"])
        if self.has_labels:
            item["labels"] = torch.tensor(int(row["label_bin"]), dtype=torch.long)
        return item

def build_weighted_sampler(y: np.ndarray):
    y = np.asarray(y).astype(int)
    class_counts = np.bincount(y, minlength=2)
    weights_per_class = 1.0 / np.maximum(class_counts, 1)
    sample_weights = weights_per_class[y]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

def make_loaders(train_df, dev_df, batch_size=BATCH_SIZE):
    train_ds = PCLDataset(train_df, tokenizer, max_len=MAX_LEN, has_labels=True)
    dev_ds = PCLDataset(dev_df, tokenizer, max_len=MAX_LEN, has_labels=True)

    if USE_WEIGHTED_SAMPLER:
        sampler = build_weighted_sampler(train_df["label_bin"].values)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    dev_loader = DataLoader(dev_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, dev_loader

train_loader, dev_loader = make_loaders(train_df, dev_df)
print("Train batches:", len(train_loader))
print("Dev batches  :", len(dev_loader))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Train batches: 524
Dev batches  : 131


## 6) Model + Focal Loss

This section implements the **pretrained RoBERTa-base classifier backbone** and the **imbalance-aware Focal Loss** proposed in the report.

- **Backbone:** RoBERTa-base encoder loaded from a pretrained checkpoint
- **Our contribution:** replacing standard cross-entropy with **Focal Loss** for imbalanced binary PCL detection


In [14]:
class RobertaBinaryClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=DROPOUT):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # RoBERTa has no token_type_ids; use CLS representation
        cls_repr = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_repr))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {"loss": loss, "logits": logits}

class FocalLoss(nn.Module):
    # Binary classification implemented on 2-class logits
    def __init__(self, alpha=0.75, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # CE per-sample
        ce = nn.functional.cross_entropy(logits, targets, reduction="none")
        probs = nn.functional.softmax(logits, dim=-1)
        pt = probs[torch.arange(len(targets), device=targets.device), targets]
        # alpha weighting: emphasize positive class (class 1)
        alpha_t = torch.where(targets == 1, torch.tensor(self.alpha, device=targets.device), torch.tensor(1.0 - self.alpha, device=targets.device))
        loss = alpha_t * ((1 - pt) ** self.gamma) * ce
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


## 7) Train / Eval Loops

This section turns the modelling choices into training/evaluation procedures:
- supervised fine-tuning of the RoBERTa-based classifier
- dev-set evaluation using **positive-class F1**
- **threshold search** utilities (used later for threshold optimisation)


In [15]:
@torch.no_grad()
def evaluate_model(model, loader, threshold=0.5, loss_fn=None):
    model.eval()
    all_logits, all_labels, all_parids = [], [], []
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        labels = batch["labels"].to(device)
        par_ids = list(batch["par_id"])
        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
        }
        out = model(**inputs)
        logits = out["logits"]

        if loss_fn is None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        else:
            loss = loss_fn(logits, labels)

        total_loss += float(loss.item())
        n_batches += 1

        all_logits.append(logits.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())
        all_parids.extend(par_ids)

    logits_np = np.vstack(all_logits)
    labels_np = np.concatenate(all_labels)
    probs = softmax_np(logits_np)[:, 1]

    metric_dict, preds = metrics_at_threshold(labels_np, probs, threshold)
    metric_dict["loss"] = total_loss / max(n_batches, 1)
    return metric_dict, {
        "par_id": all_parids,
        "labels": labels_np,
        "logits": logits_np,
        "p_pos": probs,
        "preds": preds,
    }

def train_one_seed(seed: int, train_df: pd.DataFrame, dev_df: pd.DataFrame, save_prefix: str):
    seed_everything(seed)
    model = RobertaBinaryClassifier(model_name=MODEL_NAME, dropout=DROPOUT).to(device)

    train_loader, dev_loader = make_loaders(train_df, dev_df)

    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    loss_fn = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA) if USE_FOCAL_LOSS else nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    best_dev = {"epoch": -1, "f1_pcl": -1.0, "threshold": 0.5}
    ckpt_path = CKPT_DIR / f"{save_prefix}_seed{seed}.pt"

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        step_count = 0

        for step, batch in enumerate(train_loader, start=1):
            labels = batch["labels"].to(device)
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }

            with torch.amp.autocast('cuda', enabled=USE_AMP):
                out = model(**inputs)
                logits = out["logits"]
                loss = loss_fn(logits, labels)
                loss = loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if step % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            running_loss += float(loss.item()) * GRAD_ACCUM_STEPS
            step_count += 1

        # Evaluate with default threshold first, then optimise threshold on dev probs
        dev_metrics_05, dev_cache = evaluate_model(model, dev_loader, threshold=0.5, loss_fn=loss_fn)
        best_thresh = optimise_threshold(dev_cache["labels"], dev_cache["p_pos"])
        dev_metrics_opt, _ = metrics_at_threshold(dev_cache["labels"], dev_cache["p_pos"], best_thresh["threshold"])

        print(
            f"[seed {seed}] epoch {epoch}/{NUM_EPOCHS} | "
            f"train_loss={running_loss/max(step_count,1):.4f} | "
            f"dev_f1@0.5={dev_metrics_05['f1_pcl']:.4f} | "
            f"dev_f1@opt={dev_metrics_opt['f1_pcl']:.4f} (thr={best_thresh['threshold']:.3f})"
        )

        if dev_metrics_opt["f1_pcl"] > best_dev["f1_pcl"]:
            best_dev = {
                "epoch": epoch,
                "f1_pcl": dev_metrics_opt["f1_pcl"],
                "threshold": float(best_thresh["threshold"]),
                "precision_pcl": dev_metrics_opt["precision_pcl"],
                "recall_pcl": dev_metrics_opt["recall_pcl"],
                "macro_f1": dev_metrics_opt["macro_f1"],
            }
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "seed": seed,
                    "epoch": epoch,
                    "best_dev": best_dev,
                    "config": {
                        "model_name": MODEL_NAME,
                        "max_len": MAX_LEN,
                        "dropout": DROPOUT,
                        "use_focal_loss": USE_FOCAL_LOSS,
                        "focal_alpha": FOCAL_ALPHA,
                        "focal_gamma": FOCAL_GAMMA,
                    }
                },
                ckpt_path
            )

    print(f"Best for seed {seed}: {best_dev}")
    return str(ckpt_path), best_dev


## 8) Train Multiple Seeds (Ensemble Members)

This section implements **seed ensembling** from the report:
- train the same model configuration under multiple random seeds
- save per-seed checkpoints and validation predictions
- summarise per-seed variability (to reduce reliance on a lucky single run)


In [16]:
seed_ckpts = []
seed_summaries = []

for seed in SEEDS:
    ckpt, summary = train_one_seed(seed, train_df, dev_df, save_prefix="roberta_focal")
    seed_ckpts.append(ckpt)
    seed_summaries.append({"seed": seed, **summary, "ckpt": ckpt})

seed_summary_df = pd.DataFrame(seed_summaries).sort_values("f1_pcl", ascending=False)
print("Per-seed results:")
display(seed_summary_df)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[seed 42] epoch 1/3 | train_loss=0.0384 | dev_f1@0.5=0.5224 | dev_f1@opt=0.5537 (thr=0.435)
[seed 42] epoch 2/3 | train_loss=0.0241 | dev_f1@0.5=0.5916 | dev_f1@opt=0.5980 (thr=0.485)
[seed 42] epoch 3/3 | train_loss=0.0128 | dev_f1@0.5=0.6038 | dev_f1@opt=0.6081 (thr=0.510)
Best for seed 42: {'epoch': 3, 'f1_pcl': 0.6080760095011877, 'threshold': 0.5099999999999999, 'precision_pcl': 0.5765765765765766, 'recall_pcl': 0.6432160804020101, 'macro_f1': 0.7821372879998638}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[seed 52] epoch 1/3 | train_loss=0.0368 | dev_f1@0.5=0.5483 | dev_f1@opt=0.5751 (thr=0.530)
[seed 52] epoch 2/3 | train_loss=0.0219 | dev_f1@0.5=0.5882 | dev_f1@opt=0.5976 (thr=0.515)
[seed 52] epoch 3/3 | train_loss=0.0106 | dev_f1@0.5=0.6096 | dev_f1@opt=0.6210 (thr=0.465)
Best for seed 52: {'epoch': 3, 'f1_pcl': 0.6210268948655256, 'threshold': 0.4649999999999999, 'precision_pcl': 0.6047619047619047, 'recall_pcl': 0.6381909547738693, 'macro_f1': 0.7900053765145305}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_594/3114809063.py:93: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.

[seed 62] epoch 1/3 | train_loss=0.0385 | dev_f1@0.5=0.5090 | dev_f1@opt=0.5472 (thr=0.575)
[seed 62] epoch 2/3 | train_loss=0.0247 | dev_f1@0.5=0.5700 | dev_f1@opt=0.5744 (thr=0.530)
[seed 62] epoch 3/3 | train_loss=0.0129 | dev_f1@0.5=0.5849 | dev_f1@opt=0.5963 (thr=0.575)
Best for seed 62: {'epoch': 3, 'f1_pcl': 0.5963060686015831, 'threshold': 0.575, 'precision_pcl': 0.6277777777777778, 'recall_pcl': 0.5678391959798995, 'macro_f1': 0.7780690227492032}
Per-seed results:


,seed,epoch,f1_pcl,threshold,precision_pcl,recall_pcl,macro_f1,ckpt
1,52,3,0.621027,0.465,0.604762,0.638191,0.790005,/content/drive/MyDrive/COMP60035 nlp/BestModel...
0,42,3,0.608076,0.510,0.576577,0.643216,0.782137,/content/drive/MyDrive/COMP60035 nlp/BestModel...
2,62,3,0.596306,0.575,0.627778,0.567839,0.778069,/content/drive/MyDrive/COMP60035 nlp/BestModel...


## 9) Ensemble on Official Dev + Global Threshold Optimisation

This section implements the final inference strategy:
1. load all seed checkpoints
2. **average predicted probabilities** (seed ensemble)
3. choose the dev threshold $\tau^*$ that maximises **positive-class F1**
4. reuse the selected threshold for final prediction export


In [17]:
def load_model_ckpt(ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location=device)
    config = AutoConfig.from_pretrained(MODEL_NAME)
    model = RobertaBinaryClassifier.__new__(RobertaBinaryClassifier)
    nn.Module.__init__(model)
    model.config = config
    model.encoder = AutoModel.from_config(config)
    model.dropout = nn.Dropout(DROPOUT)
    model.classifier = nn.Linear(config.hidden_size, 2)
    model = model.to(device)
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    if missing or unexpected:
        print("Checkpoint load warning | missing:", missing, "| unexpected:", unexpected)
    model.eval()
    return model, ckpt

@torch.no_grad()
def predict_probs_df(model, df: pd.DataFrame, has_labels: bool = True):
    ds = PCLDataset(df, tokenizer, max_len=MAX_LEN, has_labels=has_labels)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    all_parids, all_probs, all_logits = [], [], []
    all_labels = []
    for batch in loader:
        par_ids = list(batch["par_id"])
        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
        }
        out = model(**inputs)
        logits = out["logits"].detach().cpu().numpy()
        probs = softmax_np(logits)[:, 1]

        all_parids.extend(par_ids)
        all_probs.extend(probs.tolist())
        all_logits.append(logits)

        if has_labels:
            all_labels.extend(batch["labels"].cpu().numpy().tolist())

    out_df = pd.DataFrame({
        "par_id": all_parids,
        "p_pos": all_probs,
    })
    if has_labels:
        out_df["label_bin"] = all_labels
    return out_df

def ensemble_predict(df: pd.DataFrame, ckpt_paths: List[str], has_labels: bool = True):
    member_dfs = []
    for i, ckpt_path in enumerate(ckpt_paths):
        model, ckpt = load_model_ckpt(ckpt_path)
        pred_df = predict_probs_df(model, df, has_labels=has_labels)
        pred_df = pred_df.rename(columns={"p_pos": f"p_pos_seed{i}"})
        member_dfs.append(pred_df)

    merged = member_dfs[0].copy()
    for md in member_dfs[1:]:
        cols = ["par_id"] + [c for c in md.columns if c.startswith("p_pos_seed")]
        if has_labels and "label_bin" in md.columns:
            cols += ["label_bin"]
        merged = merged.merge(md[cols], on="par_id", how="inner")

    # if duplicated label columns appear, keep one
    if has_labels:
        label_cols = [c for c in merged.columns if c.startswith("label_bin")]
        if len(label_cols) > 1:
            first = label_cols[0]
            for c in label_cols[1:]:
                assert (merged[first] == merged[c]).all()
            merged = merged.drop(columns=label_cols[1:])
            merged = merged.rename(columns={first: "label_bin"})
        elif label_cols == ["label_bin"]:
            pass

    seed_prob_cols = [c for c in merged.columns if c.startswith("p_pos_seed")]
    merged["p_pos_ensemble"] = merged[seed_prob_cols].mean(axis=1)
    return merged

ensemble_dev = ensemble_predict(dev_df, seed_ckpts, has_labels=True)
display(ensemble_dev.head())

best_ensemble_thresh = optimise_threshold(
    ensemble_dev["label_bin"].values,
    ensemble_dev["p_pos_ensemble"].values,
    grid=THRESHOLD_GRID
)
dev_metrics_final, dev_preds = metrics_at_threshold(
    ensemble_dev["label_bin"].values,
    ensemble_dev["p_pos_ensemble"].values,
    best_ensemble_thresh["threshold"]
)

print("Best ensemble threshold on official dev:")
print(best_ensemble_thresh)
print("Final dev metrics at optimized threshold:")
print(dev_metrics_final)

print("\nClassification report (official dev, ensemble):")
print(classification_report(
    ensemble_dev["label_bin"].values,
    dev_preds,
    digits=4,
    zero_division=0
))


,par_id,p_pos_seed0,label_bin,p_pos_seed1,p_pos_seed2,p_pos_ensemble
0,4046,0.183517,1,0.157705,0.171162,0.170795
1,1279,0.936721,1,0.903843,0.869783,0.903449
2,8330,0.114490,1,0.084516,0.071026,0.090011
3,4063,0.895823,1,0.920858,0.848930,0.888537
4,4089,0.298624,1,0.272619,0.609318,0.393520


Best ensemble threshold on official dev:
{'threshold': 0.4649999999999999, 'f1_pcl': 0.6107226107226107, 'precision': 0.5695652173913044, 'recall': 0.6582914572864321}
Final dev metrics at optimized threshold:
{'threshold': 0.4649999999999999, 'precision_pcl': 0.5695652173913044, 'recall_pcl': 0.6582914572864321, 'f1_pcl': 0.6107226107226107, 'macro_f1': 0.7831479507457161}

Classification report (official dev, ensemble):
              precision    recall  f1-score   support

           0     0.9635    0.9478    0.9556      1895
           1     0.5696    0.6583    0.6107       199

    accuracy                         0.9202      2094
   macro avg     0.7665    0.8030    0.7831      2094
weighted avg     0.9261    0.9202    0.9228      2094



## 10) Export `dev.txt` and `test.txt`

This section converts model outputs into the official submission files.

- `dev.txt` is used to verify local performance on the official dev split
- `test.txt` is generated in the required format for the hidden-label test set


**What this cell does:** Create and export `dev.txt` predictions in the official order, plus save diagnostic CSV outputs.

In [18]:
final_threshold = float(best_ensemble_thresh["threshold"])
ensemble_dev["pred"] = (ensemble_dev["p_pos_ensemble"] >= final_threshold).astype(int)

# Preserve official dev split order, but handle missing IDs gracefully
dev_order_full = dev_split["par_id"].astype(int).tolist()
available_ids = set(ensemble_dev["par_id"].astype(int).tolist())

missing_export_ids = [pid for pid in dev_order_full if pid not in available_ids]
if missing_export_ids:
    print("Warning: Skipping dev IDs missing after merge/prediction:", missing_export_ids)

dev_order = [pid for pid in dev_order_full if pid in available_ids]

dev_export = (
    ensemble_dev
    .assign(par_id=ensemble_dev["par_id"].astype(int))
    .set_index("par_id")
    .loc[dev_order]
    .reset_index()
)

print(f"Exporting {len(dev_export)} dev predictions (original dev size = {len(dev_order_full)})")

save_predictions_txt(dev_export["par_id"].tolist(), dev_export["pred"].tolist(), RUN_DIR / "dev.txt")

display(dev_export.head(10))

Exporting 2094 dev predictions (original dev size = 2094)
Saved: /content/drive/MyDrive/COMP60035 nlp/BestModel/output/roberta_focal_threshold_seedensemble/dev.txt
Saved trace CSV: /content/drive/MyDrive/COMP60035 nlp/BestModel/output/roberta_focal_threshold_seedensemble/dev.csv


,par_id,p_pos_seed0,label_bin,p_pos_seed1,p_pos_seed2,p_pos_ensemble,pred
0,4046,0.183517,1,0.157705,0.171162,0.170795,0
1,1279,0.936721,1,0.903843,0.869783,0.903449,1
2,8330,0.114490,1,0.084516,0.071026,0.090011,0
3,4063,0.895823,1,0.920858,0.848930,0.888537,1
4,4089,0.298624,1,0.272619,0.609318,0.393520,0
5,432,0.113557,1,0.102516,0.267683,0.161252,0
6,4177,0.948605,1,0.904356,0.961262,0.938074,1
7,3963,0.948355,1,0.898070,0.929325,0.925250,1
8,2001,0.253117,1,0.204725,0.234762,0.230868,0
9,369,0.701705,1,0.620947,0.511658,0.611437,1


In [19]:
if test_df is not None:
    ensemble_test = ensemble_predict(test_df, seed_ckpts, has_labels=False)
    ensemble_test["pred"] = (ensemble_test["p_pos_ensemble"] >= final_threshold).astype(int)

    # Keep discovered test file order as loaded
    test_order = test_df["par_id"].tolist()
    test_export = ensemble_test.set_index("par_id").loc[test_order].reset_index()

    # If original string IDs exist (e.g., t_0, t_1, ...), carry them into the trace CSV
    if "par_id_orig" in test_df.columns:
        test_export = test_export.merge(
            test_df[["par_id", "par_id_orig"]],
            on="par_id",
            how="left",
            validate="one_to_one"
        )
        trace_ids = test_export["par_id_orig"].tolist()
    else:
        trace_ids = test_export["par_id"].tolist()

    save_predictions_txt(trace_ids, test_export["pred"].tolist(), RUN_DIR / "test.txt")
    display(test_export.head(10))
else:
    print("No official test split loaded from test/. Skipping test.txt export.")


Saved: /content/drive/MyDrive/COMP60035 nlp/BestModel/output/roberta_focal_threshold_seedensemble/test.txt
Saved trace CSV: /content/drive/MyDrive/COMP60035 nlp/BestModel/output/roberta_focal_threshold_seedensemble/test.csv


,par_id,p_pos_seed0,p_pos_seed1,p_pos_seed2,p_pos_ensemble,pred
0,t_0,0.006135,0.015124,0.040337,0.020532,0
1,t_1,0.310719,0.332890,0.274535,0.306048,0
2,t_2,0.028572,0.029371,0.051889,0.036611,0
3,t_3,0.056020,0.012794,0.016627,0.028480,0
4,t_4,0.008493,0.017250,0.040047,0.021930,0
5,t_5,0.347220,0.412718,0.301337,0.353758,0
6,t_6,0.352153,0.218402,0.466171,0.345575,0
7,t_7,0.019232,0.016613,0.052104,0.029316,0
8,t_9,0.079180,0.040478,0.179009,0.099556,0
9,t_10,0.046629,0.015235,0.100055,0.053973,0


## 11) Save Run Metadata (for reporting + reproducibility)

In [20]:

summary = {
    "model_name": MODEL_NAME,
    "seeds": SEEDS,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "use_focal_loss": USE_FOCAL_LOSS,
    "focal_alpha": FOCAL_ALPHA if USE_FOCAL_LOSS else None,
    "focal_gamma": FOCAL_GAMMA if USE_FOCAL_LOSS else None,
    "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
    "threshold_grid_size": int(len(THRESHOLD_GRID)),
    "best_ensemble_threshold_dev": float(best_ensemble_thresh["threshold"]),
    "official_dev_metrics": dev_metrics_final,
    "seed_summaries": seed_summaries,
    "paths": {
        "base_dir": str(BASE_DIR),
        "project_dir": str(PROJECT_DIR),
        "tsv_path": str(TSV_PATH),
        "train_csv": str(train_csv),
        "dev_csv": str(dev_csv),
        "run_dir": str(RUN_DIR),
    }
}

with open(RUN_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved run summary:", RUN_DIR / "run_summary.json")
print(json.dumps({
    "official_dev_f1_pcl": dev_metrics_final["f1_pcl"],
    "official_dev_macro_f1": dev_metrics_final["macro_f1"],
    "best_threshold": best_ensemble_thresh["threshold"]
}, indent=2))


Saved run summary: /content/drive/MyDrive/COMP60035 nlp/BestModel/output/roberta_focal_threshold_seedensemble/run_summary.json
{
  "official_dev_f1_pcl": 0.6107226107226107,
  "official_dev_macro_f1": 0.7831479507457161,
  "best_threshold": 0.4649999999999999
}
